# 00 — Dicionário de Dados e Preparação Inicial

**Pipeline de Engenharia de Dados** · Censo Escolar da Educação Básica 2025

Este notebook executa o pipeline inicial de ingestão e preparação:
1. Carregamento da tabela de escolas do Censo Escolar 2025
2. Recorte geográfico para o **Estado do Ceará**
3. Identificação e eliminação de **variáveis constantes** (sem variabilidade estatística no Ceará)
4. Materialização da base em formato colunar de alta performance (`.parquet`)
5. Construção automática do **Dicionário de Dados** mapeando tipos e descrições oficiais do INEP
6. Exportação dos artefatos para a pasta `data/processed/`

In [13]:
# 1. Importações
import os
import pandas as pd
import numpy as np

In [14]:
# 2. Carregamento da base bruta nacional do Censo Escolar 2025
caminho_dados = r"C:\Users\emers\Downloads\microdados_censo_escolar_2025_\microdados_censo_escolar_2025_v2\dados\Tabela_Escola_2025_V2.csv"

print("Carregando dados brutos do Censo Escolar 2025...")
df = pd.read_csv(caminho_dados, sep=';', encoding='latin1', low_memory=False)
print(f"Volumetria nacional carregada: {df.shape[0]:,} escolas e {df.shape[1]} colunas.")
df.head(3)

Carregando dados brutos do Censo Escolar 2025...
Volumetria nacional carregada: 214,192 escolas e 290 colunas.


,NU_ANO_CENSO,NO_REGIAO,CO_REGIAO,NO_UF,SG_UF,CO_UF,NO_MUNICIPIO,CO_MUNICIPIO,NO_REGIAO_GEOG_INTERM,CO_REGIAO_GEOG_INTERM,...,IN_ESP_EXCLUSIVA_MEDIO_FIC,IN_ESP_EXCLUSIVA_MEDIO_NORMAL,IN_COMUM_EJA_FUND,IN_COMUM_EJA_MEDIO,IN_COMUM_EJA_PROF,IN_ESP_EXCLUSIVA_EJA_FUND,IN_ESP_EXCLUSIVA_EJA_MEDIO,IN_ESP_EXCLUSIVA_EJA_PROF,IN_COMUM_PROF,IN_ESP_EXCLUSIVA_PROF
0,2025,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Ji-Paraná,1102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2025,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Ji-Paraná,1102.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2025,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Ji-Paraná,1102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# 3. Recorte geográfico: Filtrando apenas o Estado do Ceará (CE)
total_brasil = len(df)
df = df[df['SG_UF'] == 'CE'].reset_index(drop=True)
total_ceara = len(df)

print(f"Total Brasil: {total_brasil:,} escolas")
print(f"Total Ceará:  {total_ceara:,} escolas ({total_ceara/total_brasil:.1%} do total nacional)")
df.head(3)

Total Brasil: 214,192 escolas
Total Ceará:  10,122 escolas (4.7% do total nacional)


,NU_ANO_CENSO,NO_REGIAO,CO_REGIAO,NO_UF,SG_UF,CO_UF,NO_MUNICIPIO,CO_MUNICIPIO,NO_REGIAO_GEOG_INTERM,CO_REGIAO_GEOG_INTERM,...,IN_ESP_EXCLUSIVA_MEDIO_FIC,IN_ESP_EXCLUSIVA_MEDIO_NORMAL,IN_COMUM_EJA_FUND,IN_COMUM_EJA_MEDIO,IN_COMUM_EJA_PROF,IN_ESP_EXCLUSIVA_EJA_FUND,IN_ESP_EXCLUSIVA_EJA_MEDIO,IN_ESP_EXCLUSIVA_EJA_PROF,IN_COMUM_PROF,IN_ESP_EXCLUSIVA_PROF
0,2025,Nordeste,2,Ceará,CE,23,Abaiara,2300101,Juazeiro do Norte,2304.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2025,Nordeste,2,Ceará,CE,23,Abaiara,2300101,Juazeiro do Norte,2304.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2025,Nordeste,2,Ceará,CE,23,Abaiara,2300101,Juazeiro do Norte,2304.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
# 4. Remoção de variáveis constantes no Ceará
# Identifica colunas com 0 variância (apenas 1 valor único ou 100% nulas)
cols_constantes = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]

print(f"Total de colunas antes do corte: {df.shape[1]}")
print(f"Colunas constantes identificadas ({len(cols_constantes)}):")
for i, col in enumerate(cols_constantes, 1):
    val = df[col].dropna().unique()
    print(f"  {i:2d}. {col:32s} -> Valor fixo: {val[0] if len(val) else '100% nulo'}")

# Removendo as colunas constantes
df = df.drop(columns=cols_constantes)
print(f"\nTotal de colunas ativas restantes para análise: {df.shape[1]}")

Total de colunas antes do corte: 290
Colunas constantes identificadas (22):
   1. NU_ANO_CENSO                     -> Valor fixo: 2025
   2. NO_REGIAO                        -> Valor fixo: Nordeste
   3. CO_REGIAO                        -> Valor fixo: 2
   4. NO_UF                            -> Valor fixo: Ceará
   5. SG_UF                            -> Valor fixo: CE
   6. CO_UF                            -> Valor fixo: 23
   7. NO_SUBDISTRITO                   -> Valor fixo: 100% nulo
   8. CO_SUBDISTRITO                   -> Valor fixo: 100% nulo
   9. IN_VINCULO_SECRETARIA_SAUDE      -> Valor fixo: 0.0
  10. IN_FORMA_CONT_ES_TERMO_FOMENTO   -> Valor fixo: 0.0
  11. IN_FORMA_CONT_ES_PREST_SERV      -> Valor fixo: 0.0
  12. IN_FORMA_CONT_ES_COOP_TEC_FIN    -> Valor fixo: 0.0
  13. IN_FORMA_CONT_ES_CONSORCIO_PUB   -> Valor fixo: 0.0
  14. IN_AGUA_INEXISTENTE              -> Valor fixo: 0.0
  15. IN_ENERGIA_INEXISTENTE           -> Valor fixo: 0.0
  16. CO_LINGUA_INDIGENA_1            

In [17]:
# 5. Salvando base curada do Ceará em formato Parquet
# Isso permite que os próximos notebooks carreguem os dados em milissegundos
os.makedirs('../data/raw', exist_ok=True)
caminho_parquet = '../data/raw/escolas_ceara.parquet'
df.to_parquet(caminho_parquet, index=False)

tamanho_mb = os.path.getsize(caminho_parquet) / (1024 * 1024)
print(f"Base do Ceará salva com sucesso em: {caminho_parquet}")
print(f"Dimensões da base: {df.shape[0]} linhas x {df.shape[1]} colunas ({tamanho_mb:.2f} MB)")

Base do Ceará salva com sucesso em: ../data/raw/escolas_ceara.parquet
Dimensões da base: 10122 linhas x 268 colunas (1.05 MB)


In [18]:
# 6. Construção do Dicionário de Dados
# Carregando descrições oficiais do anexo do INEP se disponível
caminho_dicionario_inep = r"C:\Users\emers\Downloads\microdados_censo_escolar_2025_\microdados_censo_escolar_2025_v2\Anexos\ANEXO I - Dicionário de Dados\dicionário_dados_educação_básica.xlsx"
DESCRICOES = {}
if os.path.exists(caminho_dicionario_inep):
    try:
        df_raw_dic = pd.read_excel(caminho_dicionario_inep, sheet_name='Tabela_de_Escola', header=None)
        for _, row in df_raw_dic.iloc[9:].iterrows():
            var_name = str(row[1]).strip() if pd.notna(row[1]) else None
            var_desc = str(row[2]).strip() if pd.notna(row[2]) else None
            if var_name and var_desc and var_name != 'nan':
                DESCRICOES[var_name] = var_desc
    except Exception as e:
        print('Aviso ao carregar dicionário Excel:', e)

# Ajustando colunas temporais
COLS_TEMPORAIS = [c for c in ['NU_ANO_CENSO', 'DT_ANO_LETIVO_INICIO', 'DT_ANO_LETIVO_TERMINO'] if c in df.columns]

# Escalas das colunas ordinais (do menor para o maior nível)
ESCALAS_ORDINAIS = {}
COLS_ORDINAIS = list(ESCALAS_ORDINAIS)

# lista de dicionários para armazenar informações sobre as colunas
col_info = []

# percorrendo as colunas do dataframe
for col in df.columns:
    serie = df[col]
    serie_valida = serie.dropna()

    # tipo e valores (menor/maior nível, ou primeiro/último para nominais)
    if col in COLS_TEMPORAIS or col.startswith('DT_') or pd.api.types.is_datetime64_any_dtype(serie):
        col_type = 'Temporal'
        values = [serie_valida.min(), serie_valida.max()] if len(serie_valida) else [None, None]
    elif col in COLS_ORDINAIS:
        col_type = 'Ordinal'
        escala = ESCALAS_ORDINAIS[col]
        if escala is None:
            values = [serie_valida.min(), serie_valida.max()] if len(serie_valida) else [None, None]
        else:
            presentes = [nivel for nivel in escala if nivel in set(serie_valida)]
            values = [presentes[0], presentes[-1]] if presentes else [None, None]
    elif col.startswith(('IN_', 'TP_', 'CO_', 'NO_', 'SG_')):
        col_type = 'Nominal'
        uniques = serie_valida.unique()
        values = [uniques[0], uniques[-1]] if len(uniques) else [None, None]
    elif col.startswith('QT_') or pd.api.types.is_integer_dtype(serie):
        col_type = 'Discreta'
        values = [serie_valida.min(), serie_valida.max()] if len(serie_valida) else [None, None]
    elif pd.api.types.is_float_dtype(serie):
        col_type = 'Contínua'
        values = [serie_valida.min(), serie_valida.max()] if len(serie_valida) else [None, None]
    else:
        col_type = 'Nominal'
        uniques = serie_valida.unique()
        values = [uniques[0], uniques[-1]] if len(uniques) else [None, None]

    # descrição (obtida do dicionário oficial do INEP ou gerada pelo nome da coluna)
    description = DESCRICOES.get(col, f'Coluna {col}')

    # adicionando informações à lista de dicionários
    col_info.append({
        'variavel': col,
        'descricao': description,
        'valores': values,
        'Tipo': col_type
    })

# transformando lista de dicionários em dataframe
col_info_df = pd.DataFrame(col_info)
col_info_df.head(10)

,variavel,descricao,valores,Tipo
0,NO_MUNICIPIO,Nome do Município,"[Abaiara, Viçosa do Ceará]",Nominal
1,CO_MUNICIPIO,Código do Município,"[2300101, 2314102]",Nominal
2,NO_REGIAO_GEOG_INTERM,Nome da Região Geográfica Intermediária,"[Juazeiro do Norte, Crateús]",Nominal
3,CO_REGIAO_GEOG_INTERM,Código da Região Geográfica Intermediária,"[2304.0, 2305.0]",Nominal
4,NO_REGIAO_GEOG_IMED,Nome da Região Geográfica Imediata,"[Brejo Santo, São Benedito - Ipu - Guaraciaba ...",Nominal
5,CO_REGIAO_GEOG_IMED,Código da Região Geográfica Imediata,"[230012.0, 230016.0]",Nominal
6,NO_DISTRITO,Divisão Intramunicipal - Nome do Distrito,"[Abaiara, Quatiguaba]",Nominal
7,CO_DISTRITO,Divisão Intramunicipal - Código do Distrito,"[230010105, 231410225]",Nominal
8,NO_ENTIDADE,Nome da Escola,"[CEL ADAUTO BEZERRA EM, MANOEL JOSE BORGES CEI]",Nominal
9,CO_ENTIDADE,Código da Escola,"[23168730, 23563982]",Nominal


In [19]:
# 7. Resumo do Dicionário de Dados gerado
print(f"Total de variáveis catalogadas: {len(col_info_df)}")
print("\nDistribuição por tipo de variável:")
print(col_info_df['Tipo'].value_counts())

col_info_df.sample(15, random_state=42)

Total de variáveis catalogadas: 268

Distribuição por tipo de variável:
Tipo
Nominal     232
Discreta     32
Temporal      2
Contínua      2
Name: count, dtype: int64


,variavel,descricao,valores,Tipo
115,IN_SALA_OFICINAS_EDUC_PROF,Dependências físicas existentes e utilizadas n...,"[0.0, 1.0]",Nominal
213,IN_RESERVA_RENDA,Reserva de vagas por sistema de cotas para gru...,"[0.0, 1.0]",Nominal
22,IN_PODER_PUBLICO_PARCERIA,Parceria ou convênio com o poder público (parc...,"[0.0, 1.0]",Nominal
132,IN_ACESSIBILIDADE_SINAL_VISUAL,Recursos de acessibilidade para pessoas com de...,"[0.0, 1.0]",Nominal
117,IN_SALA_LEITURA,Dependências físicas existentes e utilizadas n...,"[0.0, 1.0]",Nominal
159,QT_DESKTOP_ALUNO,Quantidade de computadores em uso pelos alunos...,"[0.0, 88888.0]",Discreta
234,IN_EDUC_AMB_NENHUMA,Informe de qual(quais) forma(s) a educação amb...,"[9.0, 1.0]",Nominal
137,QT_SALAS_UTILIZADAS,Número de salas de aula utilizadas na escola (...,"[1.0, 130.0]",Discreta
30,IN_FORMA_CONT_MU_TERMO_COLAB,Forma(s) de contratação da parceria ou convêni...,"[0.0, 1.0]",Nominal
67,IN_ENERGIA_REDE_PUBLICA,Abastecimento de energia elétrica - Rede pública,"[1.0, 0.0]",Nominal


In [20]:
# 8. Salvando dicionario.csv e escala.csv
os.makedirs('../data/processed', exist_ok=True)

# Salvar dicionario.csv
col_info_df_to_save = col_info_df.copy()
col_info_df_to_save["valores"] = col_info_df_to_save["valores"].apply(
    lambda valores: [valor.item() if isinstance(valor, np.generic) else valor for valor in valores]
)
col_info_df_to_save.to_csv('../data/processed/dicionario.csv', index=False, sep=';', encoding='utf-8')
print("Dicionário salvo com sucesso em: ../data/processed/dicionario.csv")

# Salvar escala.csv
linhas_escala = []
for coluna, niveis in ESCALAS_ORDINAIS.items():
    if niveis is None:
        niveis = list(range(1, 11))
    for ordem, nivel in enumerate(niveis, start=1):
        linhas_escala.append({'coluna': coluna, 'nivel': nivel, 'ordem': ordem})

escala_df = pd.DataFrame(linhas_escala)
if not escala_df.empty:
    escala_df.to_csv('../data/processed/escala.csv', index=False, encoding='utf-8', sep=';')
    print("Escalas salvas em: ../data/processed/escala.csv")
else:
    pd.DataFrame(columns=['coluna', 'nivel', 'ordem']).to_csv('../data/processed/escala.csv', index=False, encoding='utf-8', sep=';')
    print("Nenhuma escala ordinal configurada. Arquivo escala.csv inicializado vazio.")

Dicionário salvo com sucesso em: ../data/processed/dicionario.csv
Nenhuma escala ordinal configurada. Arquivo escala.csv inicializado vazio.


In [22]:
df.NO_DISTRITO.value_counts()

NO_DISTRITO
Fortaleza              1403
Maracanaú               292
Juazeiro do Norte       222
Caucaia                 193
Sobral                  160
                       ... 
Munguba                   1
São Pedro do Gavião       1
Pio X                     1
Retiro                    1
Calabaça                  1
Name: count, Length: 751, dtype: int64